# 02 · confidence 기반 병렬 복원

각 round에서 확신이 높은 위치를 동시에 확정하는 toy decoder다. 실제 neural probability나 논문 품질·속도를 재현하지 않는다.

**학습 목표**: confidence threshold가 병렬 확정 위치 수와 decoding round 수에 미치는 영향을 비교한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 외부 패키지는 필요 없다.

In [ ]:
# 함수로 정책을 감싸 동일 schedule에서 threshold만 바꾸어 공정하게 비교한다.
def confidence_decode(schedule, threshold):
    output = ['[MASK]'] * len(schedule[0])
    trace = []
    for round_id, predictions in enumerate(schedule, start=1):
        candidates = [(i, tok, conf) for i, (tok, conf) in enumerate(predictions)
                      if output[i] == '[MASK]']
        accepted = [(i, tok, conf) for i, tok, conf in candidates if conf >= threshold]
        if not accepted and candidates:
            accepted = [max(candidates, key=lambda item: item[2])]
        for i, token, _ in accepted:
            output[i] = token
        trace.append((round_id, accepted, output.copy()))
        if '[MASK]' not in output:
            break
    return output, trace


In [ ]:
schedule = [
    [('A', .98), ('B', .62), ('C', .91), ('D', .55)],
    [('A', .99), ('B', .84), ('C', .97), ('D', .73)],
    [('A', .99), ('B', .94), ('C', .99), ('D', .92)],
]

for threshold in (0.70, 0.90, 0.92):
    result, trace = confidence_decode(schedule, threshold)
    assert '[MASK]' not in result
    print(f'threshold={threshold:.2f}: rounds={len(trace)}, result={result}')
    for round_id, accepted, state in trace:
        print(' ', round_id, [(i, round(c, 2)) for i, _, c in accepted], state)


낮은 threshold는 더 많은 토큰을 일찍 확정하지만 오확정 위험이 커진다. 실제 구현은 confidence calibration, EOS, block 경계와 cache 일관성을 함께 검증해야 한다.